In [205]:
import numpy as np
import pandas as pd
import mne
import math

In [485]:
offline = pd.read_csv("thetaPSD_online_2back_-1580742246_20260602_130048.csv")
offline

,Time,Ch01,Ch02,Ch03,Ch04,Ch05
0,0.000,-10323.453125,-0.016601,0.000276,0.000006,136.921722
1,0.004,-11039.997070,-0.145876,0.021280,0.000431,8084.414062
2,0.008,-10958.791992,-0.647873,0.419739,0.008826,16897.099609
3,0.012,-10429.335938,-1.981727,3.927240,0.087371,163496.703125
4,0.016,-10293.412109,-4.767033,22.724600,0.541863,163496.703125
...,...,...,...,...,...,...
9909,39.636,-9271.359375,1.957684,3.832525,2.517561,2.517561
9910,39.640,-8864.833008,1.735525,3.012046,2.535042,2.517561
9911,39.644,-9268.309570,1.466872,2.151714,2.538392,2.517561
9912,39.648,-10124.755859,1.151984,1.327068,2.531501,2.517561


In [486]:
online = pd.read_csv("thetaPSD_online_2back_-1580742246_20260602_130048.csv")
online

,Time,Ch01,Ch02,Ch03,Ch04,Ch05
0,0.000,-10323.453125,-0.016601,0.000276,0.000006,136.921722
1,0.004,-11039.997070,-0.145876,0.021280,0.000431,8084.414062
2,0.008,-10958.791992,-0.647873,0.419739,0.008826,16897.099609
3,0.012,-10429.335938,-1.981727,3.927240,0.087371,163496.703125
4,0.016,-10293.412109,-4.767033,22.724600,0.541863,163496.703125
...,...,...,...,...,...,...
9909,39.636,-9271.359375,1.957684,3.832525,2.517561,2.517561
9910,39.640,-8864.833008,1.735525,3.012046,2.535042,2.517561
9911,39.644,-9268.309570,1.466872,2.151714,2.538392,2.517561
9912,39.648,-10124.755859,1.151984,1.327068,2.531501,2.517561


In [487]:
theta_z_csv = pd.read_csv("theta_z_values_2back_-1580742246.csv", header= None, names = ["Time", "online z_score"])
theta_z_csv

,Time,online z_score
0,11.910610,0.641425
1,11.912054,0.641425
2,11.912468,0.641425
3,11.918230,0.641425
4,11.928654,0.641425
...,...,...
7689,47.573146,7.699154
7690,47.573285,7.699154
7691,47.573504,7.699154
7692,47.577728,7.699154


In [488]:
merged_df = pd.merge_asof(
    left=offline.sort_values("Time"),
    right=theta_z_csv.sort_values("Time"),
    on="Time",
    direction="nearest",
    tolerance=0.01   # 50 ms
)
merged_df

,Time,Ch01,Ch02,Ch03,Ch04,Ch05,online z_score
0,0.000,-10323.453125,-0.016601,0.000276,0.000006,136.921722,NaN
1,0.004,-11039.997070,-0.145876,0.021280,0.000431,8084.414062,NaN
2,0.008,-10958.791992,-0.647873,0.419739,0.008826,16897.099609,NaN
3,0.012,-10429.335938,-1.981727,3.927240,0.087371,163496.703125,NaN
4,0.016,-10293.412109,-4.767033,22.724600,0.541863,163496.703125,NaN
...,...,...,...,...,...,...,...
9909,39.636,-9271.359375,1.957684,3.832525,2.517561,2.517561,0.219865
9910,39.640,-8864.833008,1.735525,3.012046,2.535042,2.517561,0.219865
9911,39.644,-9268.309570,1.466872,2.151714,2.538392,2.517561,0.219865
9912,39.648,-10124.755859,1.151984,1.327068,2.531501,2.517561,0.219865


In [489]:
from scipy.signal import butter, lfilter

In [490]:
power.shape

(9914,)

In [491]:
from scipy.signal import iirnotch, butter, lfilter
import numpy as np

buffer = []
theta = []

SONICATION_TIME = 3
COOLDOWN_TIME = 5
THETA_THRESHOLD_Z = 1.5
MU = 2.32
SIGMA = 4.18
MAD_THRESHOLD = 6
INITIAL_CUTOFF = 100.0
BUFFER_SIZE = 500

fs = 250.0

# filtering

raw = merged_df[" Ch01"].values.astype(float)

# Notch
b_notch, a_notch = iirnotch(60/(fs/2), Q=30)
notched = lfilter(b_notch, a_notch, raw)

# Bandpass 4–7 Hz
b_bp, a_bp = butter(4, [4/(fs/2), 7/(fs/2)], btype='band')
theta_filt = lfilter(b_bp, a_bp, notched)

# Power
power = theta_filt ** 2

# Moving average
smooth = np.convolve(power, np.ones(50)/50, mode='same')

# Decimate by 10
decimated = smooth[::10]

# Hold operator: repeat each decimated value 10×
held = np.repeat(decimated, 10)
held = held[:len(raw)]   # match length

# offline loop

for i in range(len(raw)):

    theta_val = held[i]   # *** scalar, correct ***

    # Warm-up
    if len(buffer) <= 450:
        if theta_val < INITIAL_CUTOFF:
            buffer.append(theta_val)
        theta.append(np.nan)
        continue

    # Rolling buffer
    if len(buffer) > BUFFER_SIZE:
        buffer.pop(0)

    arr = np.array(buffer)
    median = np.median(arr)
    mad = np.median(np.abs(arr - median)) + 1e-6

    # Artifact rejection
    z_art = abs(theta_val - median) / mad
    if z_art > MAD_THRESHOLD:
        theta.append(np.nan)
        continue

    # Clean sample
    buffer.append(theta_val)

    # Z-score
    theta_z = abs(theta_val - MU) / SIGMA
    theta.append(theta_z)


In [492]:
merged_df["offline z-score"] = theta
merged_df["online z_score"] = merged_df["online z_score"].round(2)
merged_df["offline z-score"] = merged_df["offline z-score"].round(2)
merged_df

,Time,Ch01,Ch02,Ch03,Ch04,Ch05,online z_score,offline z-score
0,0.000,-10323.453125,-0.016601,0.000276,0.000006,136.921722,NaN,NaN
1,0.004,-11039.997070,-0.145876,0.021280,0.000431,8084.414062,NaN,NaN
2,0.008,-10958.791992,-0.647873,0.419739,0.008826,16897.099609,NaN,NaN
3,0.012,-10429.335938,-1.981727,3.927240,0.087371,163496.703125,NaN,NaN
4,0.016,-10293.412109,-4.767033,22.724600,0.541863,163496.703125,NaN,NaN
...,...,...,...,...,...,...,...,...
9909,39.636,-9271.359375,1.957684,3.832525,2.517561,2.517561,0.22,0.01
9910,39.640,-8864.833008,1.735525,3.012046,2.535042,2.517561,0.22,0.15
9911,39.644,-9268.309570,1.466872,2.151714,2.538392,2.517561,0.22,0.15
9912,39.648,-10124.755859,1.151984,1.327068,2.531501,2.517561,0.22,0.15


In [493]:
LIFU = np.zeros(len(merged_df))
last_trigger_time = -999

for i in range(len(merged_df)):
    theta_z = merged_df["offline z-score"][i]
    now = merged_df["Time"][i]

    if theta_z > THETA_THRESHOLD_Z and theta_z < MAD_THRESHOLD:
        if now - last_trigger_time > COOLDOWN_TIME:
            LIFU[i] = 1.0
            last_trigger_time = now


In [494]:
merged_df["LIFU"] = LIFU
merged_df

,Time,Ch01,Ch02,Ch03,Ch04,Ch05,online z_score,offline z-score,LIFU
0,0.000,-10323.453125,-0.016601,0.000276,0.000006,136.921722,NaN,NaN,0.0
1,0.004,-11039.997070,-0.145876,0.021280,0.000431,8084.414062,NaN,NaN,0.0
2,0.008,-10958.791992,-0.647873,0.419739,0.008826,16897.099609,NaN,NaN,0.0
3,0.012,-10429.335938,-1.981727,3.927240,0.087371,163496.703125,NaN,NaN,0.0
4,0.016,-10293.412109,-4.767033,22.724600,0.541863,163496.703125,NaN,NaN,0.0
...,...,...,...,...,...,...,...,...,...
9909,39.636,-9271.359375,1.957684,3.832525,2.517561,2.517561,0.22,0.01,0.0
9910,39.640,-8864.833008,1.735525,3.012046,2.535042,2.517561,0.22,0.15,0.0
9911,39.644,-9268.309570,1.466872,2.151714,2.538392,2.517561,0.22,0.15,0.0
9912,39.648,-10124.755859,1.151984,1.327068,2.531501,2.517561,0.22,0.15,0.0


In [495]:
mask_lifu = merged_df["LIFU"] == 1.0
lifu_on = merged_df[mask_lifu]
lifu_on

,Time,Ch01,Ch02,Ch03,Ch04,Ch05,online z_score,offline z-score,LIFU
970,3.88,-9975.029297,4.208998,17.715664,6.268857,6.045803,NaN,1.58,1.0
5040,20.16,-8897.162109,0.669720,0.448524,4.170141,4.180262,0.28,1.78,1.0
6880,27.52,-9403.896484,-2.825197,7.981739,23.650427,24.448910,0.31,1.88,1.0
8470,33.88,-9378.123047,3.671759,13.481817,8.043950,7.927518,0.49,1.76,1.0
9840,39.36,-9146.357422,-2.640786,6.973753,151.290131,154.393677,0.22,5.97,1.0


In [186]:
#windowing
event_times = lifu_on_online['Time'].values
event_times

array([ 14.7355743,  32.7421951,  70.1726053, 106.3969381, 124.9305275])

In [187]:
pre = 2
post = 5
windows = []
for idx, event in enumerate(event_times): 
    mask = (raw['Time'] >= (event - pre)) & (raw['Time'] <= (event + post))
    window_df = raw.loc[mask].copy()
    window_df["t_rel"] = window_df["Time"] - event 
    window_df["idx"] = idx
    windows.append(window_df)
    

ValueError: picks ('Time') could not be interpreted as channel names (no channel "['Time']"), channel types (no type "Time" present), or a generic type (just "all" or "data")

In [ ]:
windows_df = pd.concat(windows, ignore_index=True)
windows_df_0 = windows_df[windows_df['idx']==0]
windows_df_0

In [ ]:
mask = merged_df[